

En esta actividad se implementa una solución completa para clasificar imágenes del dataset MNIST usando únicamente PyTorch para el modelo, entrenamiento y evaluación. También se decidió usar la CPU por que no termine de entender como usar cuda pero por si acaso hay una comprobacion en un if, ademas marco la semilla en una base solo para guardar y tener un resultado "fijo" temporlmente.

In [10]:
import gzip
from pathlib import Path
import numpy as np
import torch



torch.manual_seed(42)
np.random.seed(42)


dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo en uso:", dispositivo)

Dispositivo en uso: cpu


aqui cargo el set de imagenes, por que lo hice con la ruta absoluta que incluye mi usuario y eso es que perdi la costumbre de programar localmente y preferi hacerlo asi por si acaso, los nombres de las variables y funciones son asi en español para no confindirme en lo que hacen, se que hay que usar torch para todo pero no creo que se refiera a todo todo solo a los calculos y los esenciales, asi que por eso pues en estas partes elementos de otras librerias debido a ello mas adelante se vera reflejado el uso exclusivo de torch, use como base para esto el codigo de usted en el ejemplo

In [11]:
def obteneretiquetas(ruta):
    with gzip.open(ruta, "rb") as datos:
        etiquetas = datos.read()[8:]
        return np.frombuffer(etiquetas, dtype=np.uint8)


def obtenerimg(ruta):
    with gzip.open(ruta, "rb") as datos:
        _ = int.from_bytes(datos.read(4), "big")
        numeroimg = int.from_bytes(datos.read(4), "big")
        filas = int.from_bytes(datos.read(4), "big")
        columnas = int.from_bytes(datos.read(4), "big")
        img = datos.read()
        return np.frombuffer(img, dtype=np.uint8).reshape((numeroimg, filas, columnas))


def cargardatasetmnist(rutamnist):
    imgentrenamientovalidacion = obtenerimg(Path(rutamnist) / "train-images-idx3-ubyte.gz")
    etiquetasentrenamientovalidacion = obteneretiquetas(Path(rutamnist) / "train-labels-idx1-ubyte.gz")

    imgentrenamiento = imgentrenamientovalidacion[:50000]
    etiquetasentrenamiento = etiquetasentrenamientovalidacion[:50000]

    imgvalidacion = imgentrenamientovalidacion[50000:]
    etiquetasvalidacion = etiquetasentrenamientovalidacion[50000:]

    imgprueba = obtenerimg(Path(rutamnist) / "t10k-images-idx3-ubyte.gz")
    etiquetasprueba = obteneretiquetas(Path(rutamnist) / "t10k-labels-idx1-ubyte.gz")

    return (
        imgentrenamiento,
        etiquetasentrenamiento,
        imgvalidacion,
        etiquetasvalidacion,
        imgprueba,
        etiquetasprueba
    )


rutamnist = r"C:\Users\Rafael\Documents\GitHub\computo-inteligente-Rafael-Negrete-Leyva\mnist"

imgentrenamientonp, etiquetasentrenamientonp, imgvalidacionnp, etiquetasvalidacionnp, imgpruebanp, etiquetaspruebanp = cargardatasetmnist(rutamnist)

print("Train:", imgentrenamientonp.shape, etiquetasentrenamientonp.shape)
print("Validación:", imgvalidacionnp.shape, etiquetasvalidacionnp.shape)
print("Prueba:", imgpruebanp.shape, etiquetaspruebanp.shape)

Train: (50000, 28, 28) (50000,)
Validación: (10000, 28, 28) (10000,)
Prueba: (10000, 28, 28) (10000,)


Se normalizan los pixeles al rango de 0 a 1 para facilitar el entrenamiento. También se convierten las etiquetas a tipo long, porque esa es la forma que necesita la función de pérdida para clasificación multiclase.

In [13]:
imgentrenamiento = torch.from_numpy(imgentrenamientonp).float() / 255.0
imgvalidacion = torch.from_numpy(imgvalidacionnp).float() / 255.0
imgprueba = torch.from_numpy(imgpruebanp).float() / 255.0

etiquetasentrenamiento = torch.from_numpy(etiquetasentrenamientonp).long()
etiquetasvalidacion = torch.from_numpy(etiquetasvalidacionnp).long()
etiquetasprueba = torch.from_numpy(etiquetaspruebanp).long()

datasetentrenamiento = torch.utils.data.TensorDataset(imgentrenamiento, etiquetasentrenamiento)
datasetvalidacion = torch.utils.data.TensorDataset(imgvalidacion, etiquetasvalidacion)
datasetprueba = torch.utils.data.TensorDataset(imgprueba, etiquetasprueba)

print("Elementos entrenamiento:", len(datasetentrenamiento))
print("Elementos validación:", len(datasetvalidacion))
print("Elementos prueba:", len(datasetprueba))

Elementos entrenamiento: 50000
Elementos validación: 10000
Elementos prueba: 10000


La red se construye usando únicamente capas Linear y ReLU

In [14]:
class Redneuronalmulticlase(torch.nn.Module):
    def __init__(self, capasocultas):
        super().__init__()

        capas = []
        tamanoentrada = 28 * 28

        for tamanosalida in capasocultas:
            capas.append(torch.nn.Linear(tamanoentrada, tamanosalida))
            capas.append(torch.nn.ReLU())
            tamanoentrada = tamanosalida

        capas.append(torch.nn.Linear(tamanoentrada, 10))
        self.red = torch.nn.Sequential(*capas)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.red(x)

separaron varias funciones auxiliares hace más fácil entrenar y usar varios modelos

In [16]:
def crearcargadores(batchsizeentrenamiento, batchsizeevaluacion=1024):
    cargadorentrenamiento = torch.utils.data.DataLoader(
        datasetentrenamiento,
        batch_size=batchsizeentrenamiento,
        shuffle=True
    )

    cargadorvalidacion = torch.utils.data.DataLoader(
        datasetvalidacion,
        batch_size=batchsizeevaluacion,
        shuffle=False
    )

    cargadorprueba = torch.utils.data.DataLoader(
        datasetprueba,
        batch_size=batchsizeevaluacion,
        shuffle=False
    )

    return cargadorentrenamiento, cargadorvalidacion, cargadorprueba


def crearoptimizador(nombreoptimizador, parametrosmodelo, learningrate):
    nombre = nombreoptimizador.lower()

    if nombre == "adam":
        return torch.optim.Adam(parametrosmodelo, lr=learningrate)

    if nombre == "sgd":
        return torch.optim.SGD(parametrosmodelo, lr=learningrate, momentum=0.9)

    raise ValueError(f"Optimizador no soportado: {nombreoptimizador}")


def evaluarmodelo(modelo, cargador, funcionperdida, devolverpredicciones=False):
    modelo.eval()

    perdidaacumulada = 0.0
    totalejemplos = 0
    totalcorrectos = 0

    prediccionestotales = []
    etiquetastotales = []

    with torch.no_grad():
        for imglote, etiquetaslote in cargador:
            imglote = imglote.to(dispositivo)
            etiquetaslote = etiquetaslote.to(dispositivo)

            logits = modelo(imglote)
            loss = funcionperdida(logits, etiquetaslote)

            predicciones = torch.argmax(logits, dim=1)

            tamanolote = etiquetaslote.size(0)
            perdidaacumulada += loss.item() * tamanolote
            totalejemplos += tamanolote
            totalcorrectos += (predicciones == etiquetaslote).sum().item()

            if devolverpredicciones:
                prediccionestotales.append(predicciones.cpu())
                etiquetastotales.append(etiquetaslote.cpu())

    losspromedio = perdidaacumulada / totalejemplos
    accuracy = totalcorrectos / totalejemplos

    if devolverpredicciones:
        prediccionestotales = torch.cat(prediccionestotales)
        etiquetastotales = torch.cat(etiquetastotales)
        return losspromedio, accuracy, prediccionestotales, etiquetastotales

    return losspromedio, accuracy